In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("..")

STORE_FILE = BASE_DIR / "data" / "synthetic" / "stores_master.csv"
SALES_DIR = BASE_DIR / "data" / "synthetic"

stores = pd.read_csv(STORE_FILE)

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

sales_list = []

for file in sales_files:
    df = pd.read_csv(
        file,
        usecols=[
            "sales_line_id",
            "order_id",
            "store_id",
            "quantity",
            "unit_selling_price",
            "discount_pct"
        ]
    )
    sales_list.append(df)

sales = pd.concat(sales_list, ignore_index=True)

print("=" * 70)
print("02.11 STEP 1 — STORE / REGIONAL DISTRIBUTION ANALYSIS")
print("=" * 70)

print(f"\nStores in master: {len(stores)}")
print(f"Sales files loaded: {len(sales_files)}")
print(f"Sales rows loaded: {len(sales)}")

print("\nStore region distribution:")
print(stores["region"].value_counts().sort_index())

print("\nStore type distribution:")
print(stores["store_type"].value_counts())

print("\nSales by region:")

sales_region = (
    sales
    .merge(
        stores[["store_id", "region"]],
        on="store_id",
        how="left"
    )
    .groupby("region")
    .agg(
        sales_lines=("sales_line_id", "count"),
        quantity=("quantity", "sum"),
        revenue=("unit_selling_price", "sum")
    )
    .reset_index()
)

sales_region["sales_line_percentage"] = (
    sales_region["sales_lines"]
    / sales_region["sales_lines"].sum()
    * 100
)

sales_region["revenue_percentage"] = (
    sales_region["revenue"]
    / sales_region["revenue"].sum()
    * 100
)

print(sales_region.sort_values("revenue", ascending=False).to_string(index=False))

print("\n02.11 Step 1 analysis completed.")

02.11 STEP 1 — STORE / REGIONAL DISTRIBUTION ANALYSIS

Stores in master: 60
Sales files loaded: 32
Sales rows loaded: 2118200

Store region distribution:
region
Central     6
East        9
North      13
South      18
West       14
Name: count, dtype: int64

Store type distribution:
store_type
Neighborhood    24
Mall            18
High Street     10
Outlet           8
Name: count, dtype: int64

Sales by region:
 region  sales_lines  quantity      revenue  sales_line_percentage  revenue_percentage
  South       649016   1037210 1.998571e+10              30.639977           30.636195
   West       467798    747769 1.441180e+10              22.084695           22.091929
  North       455762    729447 1.404203e+10              21.516476           21.525099
   East       318726    509865 9.804711e+09              15.047021           15.029695
Central       226898    363484 6.991353e+09              10.711831           10.717082

02.11 Step 1 analysis completed.


In [2]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("..")

STORE_FILE = BASE_DIR / "data" / "synthetic" / "stores_master.csv"

stores = pd.read_csv(STORE_FILE)

# Approved regional demand multipliers
REGIONAL_DEMAND_MULTIPLIERS = {
    "South": 1.10,
    "West": 1.05,
    "North": 1.00,
    "East": 0.88,
    "Central": 0.82
}

# Apply regional multiplier
stores["regional_demand_multiplier"] = (
    stores["region"].map(REGIONAL_DEMAND_MULTIPLIERS)
)

# Store-size classification based on existing floor area
stores["store_size"] = pd.qcut(
    stores["floor_area_sqft"],
    q=3,
    labels=["Small", "Medium", "Large"],
    duplicates="drop"
)

# Store-size contribution weights
STORE_SIZE_WEIGHTS = {
    "Small": 0.80,
    "Medium": 1.00,
    "Large": 1.25
}

stores["store_size_weight"] = (
    stores["store_size"]
    .astype(str)
    .map(STORE_SIZE_WEIGHTS)
)

# Combined store behavior weight
stores["store_behavior_weight"] = (
    stores["regional_demand_multiplier"]
    * stores["store_size_weight"]
)

print("=" * 70)
print("02.11 STEP 2 — REGIONAL / STORE-SIZE CONTROL PLAN")
print("=" * 70)

print("\nRegional demand multipliers:")
for region, multiplier in REGIONAL_DEMAND_MULTIPLIERS.items():
    print(f"{region}: {multiplier}")

print("\nStore-size distribution:")
print(
    stores["store_size"]
    .value_counts()
    .sort_index()
)

print("\nStore behavior weight summary:")
print(
    stores[
        [
            "store_id",
            "region",
            "store_type",
            "floor_area_sqft",
            "store_size",
            "regional_demand_multiplier",
            "store_size_weight",
            "store_behavior_weight"
        ]
    ].head(10).to_string(index=False)
)

print("\nMissing regional multipliers:",
      stores["regional_demand_multiplier"].isna().sum())

print("Missing store-size weights:",
      stores["store_size_weight"].isna().sum())

print("\n02.11 Step 2 control-plan analysis completed.")

02.11 STEP 2 — REGIONAL / STORE-SIZE CONTROL PLAN

Regional demand multipliers:
South: 1.1
West: 1.05
North: 1.0
East: 0.88
Central: 0.82

Store-size distribution:
store_size
Small     20
Medium    20
Large     20
Name: count, dtype: int64

Store behavior weight summary:
  store_id  region   store_type  floor_area_sqft store_size  regional_demand_multiplier  store_size_weight  store_behavior_weight
STORE00001   South  High Street            18904      Large                        1.10               1.25                 1.3750
STORE00002   North Neighborhood            16002     Medium                        1.00               1.00                 1.0000
STORE00003 Central Neighborhood            18040      Large                        0.82               1.25                 1.0250
STORE00004    West Neighborhood            19452      Large                        1.05               1.25                 1.3125
STORE00005    West       Outlet            17054      Large                   

In [3]:
from pathlib import Path

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "store_regional_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTROL_PLAN_FILE = (
    OUTPUT_DIR
    / "store_regional_control_plan.csv"
)

control_plan = stores[
    [
        "store_id",
        "region",
        "store_type",
        "floor_area_sqft",
        "store_size",
        "regional_demand_multiplier",
        "store_size_weight",
        "store_behavior_weight"
    ]
].copy()

control_plan.to_csv(
    CONTROL_PLAN_FILE,
    index=False
)

print("=" * 70)
print("02.11 STEP 3 — STORE / REGIONAL CONTROL PLAN SAVED")
print("=" * 70)

print(f"\nStores saved: {len(control_plan)}")
print(f"Unique store IDs: {control_plan['store_id'].nunique()}")

print("\nMissing values:")
print(control_plan.isna().sum())

print("\nControl plan saved to:")
print(CONTROL_PLAN_FILE)

print("\n02.11 Step 3 completed.")

02.11 STEP 3 — STORE / REGIONAL CONTROL PLAN SAVED

Stores saved: 60
Unique store IDs: 60

Missing values:
store_id                      0
region                        0
store_type                    0
floor_area_sqft               0
store_size                    0
regional_demand_multiplier    0
store_size_weight             0
store_behavior_weight         0
dtype: int64

Control plan saved to:
..\data\synthetic\store_regional_adjusted\store_regional_control_plan.csv

02.11 Step 3 completed.


In [4]:
import pandas as pd

print("=" * 70)
print("02.11 STEP 4 — STORE / REGIONAL CONTROL PLAN VALIDATION")
print("=" * 70)

df = pd.read_csv(CONTROL_PLAN_FILE)

expected_regions = {
    "North",
    "South",
    "East",
    "West",
    "Central"
}

actual_regions = set(df["region"].dropna().unique())

duplicate_store_ids = df["store_id"].duplicated().sum()
missing_store_ids = df["store_id"].isna().sum()
missing_regions = df["region"].isna().sum()
missing_regional_weights = df["regional_demand_multiplier"].isna().sum()
missing_size_weights = df["store_size_weight"].isna().sum()
missing_behavior_weights = df["store_behavior_weight"].isna().sum()

invalid_behavior_weights = (
    df["store_behavior_weight"] <= 0
).sum()

missing_expected_regions = expected_regions - actual_regions

print(f"\nStores validated: {len(df)}")
print(f"Unique store IDs: {df['store_id'].nunique()}")

print("\nRegion validation:")
print(f"Expected regions: {expected_regions}")
print(f"Actual regions: {actual_regions}")
print(f"Missing expected regions: {missing_expected_regions}")

print("\nStore-size distribution:")
print(df["store_size"].value_counts().sort_index())

print("\nValidation counts:")
print(f"Duplicate store IDs: {duplicate_store_ids}")
print(f"Missing store IDs: {missing_store_ids}")
print(f"Missing regions: {missing_regions}")
print(f"Missing regional multipliers: {missing_regional_weights}")
print(f"Missing store-size weights: {missing_size_weights}")
print(f"Missing behavior weights: {missing_behavior_weights}")
print(f"Zero/negative behavior weights: {invalid_behavior_weights}")

if (
    len(df) == 60
    and df["store_id"].nunique() == 60
    and duplicate_store_ids == 0
    and missing_store_ids == 0
    and missing_regions == 0
    and len(missing_expected_regions) == 0
    and missing_regional_weights == 0
    and missing_size_weights == 0
    and missing_behavior_weights == 0
    and invalid_behavior_weights == 0
):
    print("\n02.11 Step 4 validation: PASSED")
else:
    print("\n02.11 Step 4 validation: REVIEW REQUIRED")

02.11 STEP 4 — STORE / REGIONAL CONTROL PLAN VALIDATION

Stores validated: 60
Unique store IDs: 60

Region validation:
Expected regions: {'South', 'East', 'North', 'West', 'Central'}
Actual regions: {'North', 'West', 'South', 'East', 'Central'}
Missing expected regions: set()

Store-size distribution:
store_size
Large     20
Medium    20
Small     20
Name: count, dtype: int64

Validation counts:
Duplicate store IDs: 0
Missing store IDs: 0
Missing regions: 0
Missing regional multipliers: 0
Missing store-size weights: 0
Missing behavior weights: 0
Zero/negative behavior weights: 0

02.11 Step 4 validation: PASSED


In [5]:
import pandas as pd

print("=" * 70)
print("02.11 STEP 5 — CURRENT VS CONTROLLED REGIONAL CONTRIBUTION")
print("=" * 70)

# Load store control plan
control = pd.read_csv(CONTROL_PLAN_FILE)

# Load all monthly sales
sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

sales_list = []

for file in sales_files:
    df = pd.read_csv(
        file,
        usecols=[
            "sales_line_id",
            "store_id",
            "quantity",
            "unit_selling_price"
        ]
    )
    sales_list.append(df)

sales = pd.concat(sales_list, ignore_index=True)

# Calculate sales value
sales["sales_value"] = (
    sales["quantity"]
    * sales["unit_selling_price"]
)

# Attach region and store behavior weight
sales_with_control = sales.merge(
    control[
        [
            "store_id",
            "region",
            "store_size",
            "regional_demand_multiplier",
            "store_size_weight",
            "store_behavior_weight"
        ]
    ],
    on="store_id",
    how="left"
)

# Regional actual contribution
regional_actual = (
    sales_with_control
    .groupby("region")
    .agg(
        sales_lines=("sales_line_id", "count"),
        sales_value=("sales_value", "sum"),
        control_weight=("store_behavior_weight", "sum")
    )
    .reset_index()
)

regional_actual["actual_sales_pct"] = (
    regional_actual["sales_lines"]
    / regional_actual["sales_lines"].sum()
    * 100
)

regional_actual["actual_revenue_pct"] = (
    regional_actual["sales_value"]
    / regional_actual["sales_value"].sum()
    * 100
)

regional_actual["control_weight_pct"] = (
    regional_actual["control_weight"]
    / regional_actual["control_weight"].sum()
    * 100
)

regional_actual["revenue_vs_control_difference_pct"] = (
    regional_actual["actual_revenue_pct"]
    - regional_actual["control_weight_pct"]
)

print("\nRegional comparison:")
print(
    regional_actual[
        [
            "region",
            "sales_lines",
            "actual_sales_pct",
            "actual_revenue_pct",
            "control_weight_pct",
            "revenue_vs_control_difference_pct"
        ]
    ]
    .sort_values("control_weight_pct", ascending=False)
    .to_string(index=False)
)

print("\nSales rows without store control mapping:",
      sales_with_control["region"].isna().sum())

print("Sales rows with missing behavior weight:",
      sales_with_control["store_behavior_weight"].isna().sum())

print("\n02.11 Step 5 analysis completed.")

02.11 STEP 5 — CURRENT VS CONTROLLED REGIONAL CONTRIBUTION

Regional comparison:
 region  sales_lines  actual_sales_pct  actual_revenue_pct  control_weight_pct  revenue_vs_control_difference_pct
  South       649016         30.639977           30.614859           34.177778                          -3.562919
   West       467798         22.084695           22.085939           22.674353                          -0.588414
  North       455762         21.516476           21.526035           21.404542                           0.121494
   East       318726         15.047021           15.053910           12.329372                           2.724538
Central       226898         10.711831           10.719257            9.413955                           1.305301

Sales rows without store control mapping: 0
Sales rows with missing behavior weight: 0

02.11 Step 5 analysis completed.


In [6]:
import pandas as pd

print("=" * 70)
print("02.11 STEP 6 — STORE-LEVEL DISTRIBUTION VALIDATION")
print("=" * 70)

# Load the store/regional control plan
control = pd.read_csv(CONTROL_PLAN_FILE)

print("\nStores analysed:", len(control))
print("Unique store IDs:", control["store_id"].nunique())

# Check required control fields
required_columns = [
    "store_id",
    "region",
    "store_size",
    "regional_demand_multiplier",
    "store_size_weight",
    "store_behavior_weight"
]

missing_columns = [
    col for col in required_columns
    if col not in control.columns
]

# Check duplicates
duplicate_store_ids = control["store_id"].duplicated().sum()

# Check missing values
missing_values = control[required_columns].isna().sum().sum()

# Check invalid weights
invalid_regional_multiplier = (
    (control["regional_demand_multiplier"] <= 0).sum()
)

invalid_store_size_weight = (
    (control["store_size_weight"] <= 0).sum()
)

invalid_behavior_weight = (
    (control["store_behavior_weight"] <= 0).sum()
)

print("\nMissing required columns:", missing_columns)
print("Duplicate store IDs:", duplicate_store_ids)
print("Missing control values:", missing_values)

print(
    "Invalid regional demand multipliers:",
    invalid_regional_multiplier
)

print(
    "Invalid store-size weights:",
    invalid_store_size_weight
)

print(
    "Invalid store behavior weights:",
    invalid_behavior_weight
)

# Store-size summary
print("\nStore-size distribution:")
print(
    control["store_size"]
    .value_counts()
    .sort_index()
)

# Regional summary
print("\nRegional store distribution:")
print(
    control["region"]
    .value_counts()
    .sort_index()
)

# Show highest and lowest weighted stores
print("\nHighest store behavior weights:")
print(
    control[
        [
            "store_id",
            "region",
            "store_size",
            "regional_demand_multiplier",
            "store_size_weight",
            "store_behavior_weight"
        ]
    ]
    .sort_values("store_behavior_weight", ascending=False)
    .head(5)
    .to_string(index=False)
)

print("\nLowest store behavior weights:")
print(
    control[
        [
            "store_id",
            "region",
            "store_size",
            "regional_demand_multiplier",
            "store_size_weight",
            "store_behavior_weight"
        ]
    ]
    .sort_values("store_behavior_weight")
    .head(5)
    .to_string(index=False)
)

# Final validation
if (
    len(missing_columns) == 0
    and duplicate_store_ids == 0
    and missing_values == 0
    and invalid_regional_multiplier == 0
    and invalid_store_size_weight == 0
    and invalid_behavior_weight == 0
):
    print("\n02.11 Step 6 validation: PASSED")
else:
    print("\n02.11 Step 6 validation: REVIEW REQUIRED")

02.11 STEP 6 — STORE-LEVEL DISTRIBUTION VALIDATION

Stores analysed: 60
Unique store IDs: 60

Missing required columns: []
Duplicate store IDs: 0
Missing control values: 0
Invalid regional demand multipliers: 0
Invalid store-size weights: 0
Invalid store behavior weights: 0

Store-size distribution:
store_size
Large     20
Medium    20
Small     20
Name: count, dtype: int64

Regional store distribution:
region
Central     6
East        9
North      13
South      18
West       14
Name: count, dtype: int64

Highest store behavior weights:
  store_id region store_size  regional_demand_multiplier  store_size_weight  store_behavior_weight
STORE00001  South      Large                         1.1               1.25                  1.375
STORE00007  South      Large                         1.1               1.25                  1.375
STORE00016  South      Large                         1.1               1.25                  1.375
STORE00014  South      Large                         1.1     

In [7]:
import pandas as pd
import numpy as np

print("=" * 70)
print("02.11 STEP 7 — STORE BEHAVIOR WEIGHT VALIDATION")
print("=" * 70)

# Load the store/regional control plan
control = pd.read_csv(CONTROL_PLAN_FILE)

# Calculate the expected behavior weight
control["expected_behavior_weight"] = (
    control["regional_demand_multiplier"]
    * control["store_size_weight"]
)

# Compare calculated vs stored behavior weight
control["weight_difference"] = (
    control["store_behavior_weight"]
    - control["expected_behavior_weight"]
).abs()

# Count mismatches
weight_mismatches = (
    control["weight_difference"] > 0.000001
).sum()

print("\nStores analysed:", len(control))

print(
    "Expected behavior weights calculated:",
    control["expected_behavior_weight"].notna().sum()
)

print(
    "Behavior-weight mismatches:",
    weight_mismatches
)

print(
    "Minimum behavior weight:",
    control["store_behavior_weight"].min()
)

print(
    "Maximum behavior weight:",
    control["store_behavior_weight"].max()
)

print("\nSample store behavior weights:")

print(
    control[
        [
            "store_id",
            "region",
            "store_size",
            "regional_demand_multiplier",
            "store_size_weight",
            "store_behavior_weight",
            "expected_behavior_weight"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

# Final validation
if (
    len(control) == 60
    and control["store_id"].nunique() == 60
    and weight_mismatches == 0
    and (control["store_behavior_weight"] > 0).all()
):
    print("\n02.11 Step 7 validation: PASSED")
else:
    print("\n02.11 Step 7 validation: REVIEW REQUIRED")

02.11 STEP 7 — STORE BEHAVIOR WEIGHT VALIDATION

Stores analysed: 60
Expected behavior weights calculated: 60
Behavior-weight mismatches: 0
Minimum behavior weight: 0.656
Maximum behavior weight: 1.375

Sample store behavior weights:
  store_id  region store_size  regional_demand_multiplier  store_size_weight  store_behavior_weight  expected_behavior_weight
STORE00001   South      Large                        1.10               1.25                 1.3750                    1.3750
STORE00002   North     Medium                        1.00               1.00                 1.0000                    1.0000
STORE00003 Central      Large                        0.82               1.25                 1.0250                    1.0250
STORE00004    West      Large                        1.05               1.25                 1.3125                    1.3125
STORE00005    West      Large                        1.05               1.25                 1.3125                    1.3125
STORE00006

In [8]:
# ============================================================
# 02.11 STEP 8 — SAVE VALIDATED STORE BEHAVIOR PLAN
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 70)
print("02.11 STEP 8 — SAVE VALIDATED STORE BEHAVIOR PLAN")
print("=" * 70)

# ------------------------------------------------------------
# 1. Create output directory
# ------------------------------------------------------------

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "store_regional_adjusted"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Reload the control plan
# ------------------------------------------------------------

control = pd.read_csv(
    CONTROL_PLAN_FILE
)

# ------------------------------------------------------------
# 3. Validate before saving
# ------------------------------------------------------------

required_columns = [
    "store_id",
    "region",
    "store_type",
    "floor_area_sqft",
    "store_size",
    "regional_demand_multiplier",
    "store_size_weight",
    "store_behavior_weight"
]

missing_columns = [
    column
    for column in required_columns
    if column not in control.columns
]

duplicate_store_ids = (
    control["store_id"].duplicated().sum()
)

missing_values = (
    control[required_columns]
    .isna()
    .sum()
    .sum()
)

invalid_behavior_weights = (
    control["store_behavior_weight"] <= 0
).sum()

# ------------------------------------------------------------
# 4. Save only if validation passes
# ------------------------------------------------------------

if (
    len(missing_columns) == 0
    and len(control) == 60
    and control["store_id"].nunique() == 60
    and duplicate_store_ids == 0
    and missing_values == 0
    and invalid_behavior_weights == 0
):

    FINAL_PLAN_FILE = (
        OUTPUT_DIR
        / "store_regional_behavior_plan.csv"
    )

    control[
        required_columns
    ].to_csv(
        FINAL_PLAN_FILE,
        index=False
    )

    print("\nStores saved:", len(control))
    print(
        "Unique store IDs:",
        control["store_id"].nunique()
    )

    print(
        "Duplicate store IDs:",
        duplicate_store_ids
    )

    print(
        "Missing values:",
        missing_values
    )

    print(
        "Invalid behavior weights:",
        invalid_behavior_weights
    )

    print("\nFinal plan saved to:")
    print(FINAL_PLAN_FILE)

    print("\n02.11 Step 8 validation: PASSED")

else:

    print("\n02.11 Step 8 validation: REVIEW REQUIRED")

    print("Missing columns:", missing_columns)
    print("Duplicate store IDs:", duplicate_store_ids)
    print("Missing values:", missing_values)
    print(
        "Invalid behavior weights:",
        invalid_behavior_weights
    )

02.11 STEP 8 — SAVE VALIDATED STORE BEHAVIOR PLAN

Stores saved: 60
Unique store IDs: 60
Duplicate store IDs: 0
Missing values: 0
Invalid behavior weights: 0

Final plan saved to:
..\data\synthetic\store_regional_adjusted\store_regional_behavior_plan.csv

02.11 Step 8 validation: PASSED
